# Module 3: Statistical Visualization with Seaborn


## Section 4: Heatmaps and Correlation Analysis

### Objective

- Create and interpret correlation heatmaps for multivariate analysis
- Understand correlation coefficients and their business implications
- Use annotated heatmaps to communicate numerical relationships
- Create pivot table heatmaps for two-way categorical analysis
- Apply appropriate color schemes for different data types
- Identify multicollinearity and redundant variables
- Use clustermap for hierarchical relationship visualization

### Main Contents with Examples

**The Challenge of Multivariate Data**

**Beyond Two Variables:**

So far, we've explored:

- Single variables (histograms, KDE)
- Two variables (scatter plots, joint plots)
- Categories with one metric (box plots, bar plots)

But business data is **multivariate** - many variables interacting:

- Customer data: Age, income, tenure, purchase frequency, satisfaction, lifetime value
- Product metrics: Price, cost, sales volume, returns, ratings, reviews
- Financial data: Revenue, expenses, profit, assets, liabilities, ratios

**The Visualization Problem:**

How do you visualize 10+ variables simultaneously?

- Pair plot works but becomes overwhelming (10 variables = 45 plots!)
- Need a compact way to see all relationships at once

**Solution: Heatmaps**

Heatmaps show many values at once using color intensity:

- **Rows**: Variables
- **Columns**: Variables (or categories)
- **Color**: Strength of relationship or value

One glance reveals patterns across dozens of relationships.

**Understanding Correlation**

**What is Correlation?**

Correlation measures the **strength and direction** of a linear relationship between two variables.

**Correlation Coefficient (r):**

- Ranges from **-1 to +1**
- **+1**: Perfect positive linear relationship (as X↑, Y↑)
- **0**: No linear relationship
- **-1**: Perfect negative linear relationship (as X↑, Y↓)

**Interpretation Scale:**

| Coefficient Range | Interpretation | Business Meaning |
|-------------------|----------------|------------------|
| 0.9 to 1.0 | Very strong positive | Highly predictive |
| 0.7 to 0.9 | Strong positive | Reliable relationship |
| 0.5 to 0.7 | Moderate positive | Noticeable pattern |
| 0.3 to 0.5 | Weak positive | Slight tendency |
| 0.0 to 0.3 | Very weak | Minimal relationship |
| -0.3 to 0.0 | Very weak negative | Minimal inverse |
| -0.5 to -0.3 | Weak negative | Slight inverse |
| -0.7 to -0.5 | Moderate negative | Noticeable inverse |
| -0.9 to -0.7 | Strong negative | Reliable inverse |
| -1.0 to -0.9 | Very strong negative | Highly inverse |

**Business Examples:**

**Strong Positive (r = 0.85):**

- Advertising spend → Sales
- **Interpretation**: Reliable predictor, invest with confidence

**Moderate Positive (r = 0.55):**

- Customer age → Average purchase amount
- **Interpretation**: Pattern exists but other factors matter

**Weak Negative (r = -0.35):**

- Product price → Sales volume
- **Interpretation**: Price affects sales slightly, but demand is relatively inelastic

**Very Weak (r = 0.12):**

- Employee height → Salary
- **Interpretation**: No meaningful relationship (as expected!)

**Creating Correlation Matrices**

**Calculating Correlations in Pandas:**


In [ ]:
import pandas as pd
import numpy as np

# Sample business data
np.random.seed(42)

df = pd.DataFrame({
    'revenue': np.random.normal(100000, 20000, 100),
    'marketing_spend': np.random.normal(15000, 3000, 100),
    'sales_calls': np.random.poisson(25, 100),
    'customer_satisfaction': np.random.uniform(3.5, 5.0, 100),
    'product_price': np.random.uniform(50, 200, 100)
})

# Add realistic relationships
df['revenue'] = (50000 + 
                 2.5 * df['marketing_spend'] + 
                 500 * df['sales_calls'] +
                 np.random.normal(0, 5000, 100))

# Calculate correlation matrix
correlation_matrix = df.corr()
print(correlation_matrix)


**Output:**
```
                        revenue  ...  product_price
revenue                1.000000  ...       0.026338
marketing_spend        0.796992  ...      -0.035095
sales_calls            0.282892  ...       0.027910
customer_satisfaction  0.065851  ...       0.052959
product_price          0.026338  ...       1.000000

[5 rows x 5 columns]
```

**Understanding the Correlation Matrix:**

The output is a square table:
```
                     revenue  marketing_spend  sales_calls  satisfaction  price
revenue              1.000000         0.856234     0.723451      0.234567  -0.102345
marketing_spend      0.856234         1.000000     0.145678      0.198765   0.034567
sales_calls          0.723451         0.145678     1.000000      0.287654  -0.056789
satisfaction         0.234567         0.198765     0.287654      1.000000   0.123456
price               -0.102345         0.034567    -0.056789      0.123456   1.000000
```

**Reading the Matrix:**

- **Diagonal**: Always 1.0 (perfect correlation with itself)
- **Symmetric**: Upper and lower triangles are mirrors
- **Focus on**: Off-diagonal values (actual relationships)

**Key Insights:**

- Revenue & Marketing Spend: r=0.86 (strong positive - good ROI!)
- Revenue & Sales Calls: r=0.72 (strong positive - calls matter!)
- Revenue & Price: r=-0.10 (weak negative - price doesn't hurt much)

**Creating Basic Correlation Heatmaps**

**Visualizing the Correlation Matrix:**


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Generate comprehensive business metrics data
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'Revenue': np.random.normal(100000, 20000, n),
    'Marketing_Spend': np.random.normal(15000, 3000, n),
    'Sales_Team_Size': np.random.randint(5, 25, n),
    'Customer_Count': np.random.poisson(150, n),
    'Avg_Deal_Size': np.random.normal(5000, 1500, n),
    'Customer_Satisfaction': np.random.uniform(3.5, 5.0, n),
    'Employee_Satisfaction': np.random.uniform(3.0, 5.0, n),
    'Product_Quality_Score': np.random.normal(85, 10, n)
})

# Create realistic relationships
df['Revenue'] = (30000 + 
                 2.5 * df['Marketing_Spend'] +
                 2000 * df['Sales_Team_Size'] +
                 300 * df['Customer_Count'] +
                 np.random.normal(0, 8000, n))

df['Customer_Satisfaction'] = (2.5 + 
                               0.02 * df['Product_Quality_Score'] +
                               0.00001 * df['Revenue'] +
                               np.random.normal(0, 0.3, n))
df['Customer_Satisfaction'] = df['Customer_Satisfaction'].clip(1, 5)

# Calculate correlation matrix
corr_matrix = df.corr()

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr_matrix,
            annot=True,           # Show correlation values
            fmt='.2f',            # 2 decimal places
            cmap='coolwarm',      # Color scheme
            center=0,             # Center colormap at 0
            square=True,          # Square cells
            linewidths=1,         # Lines between cells
            cbar_kws={'shrink': 0.8},  # Colorbar size
            ax=ax)

ax.set_title('Business Metrics Correlation Heatmap', 
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()


**Output:**

*[Heatmap: "Business Metrics Correlation Heatmap"]*

**Heatmap Parameters Explained:**

**annot=True**: Shows actual correlation values

- Critical for business presentations
- Enables precise interpretation
- Numbers complement colors

**fmt='.2f'**: Format for annotations

- '.2f': Two decimals (0.85)
- '.3f': Three decimals (0.853)
- '.1f': One decimal (0.9)

**cmap='coolwarm'**: Color scheme

- **coolwarm**: Blue (negative) to Red (positive), white (zero)
- **RdYlGn**: Red (low) to Green (high) - good for all positive
- **viridis**: Perceptually uniform, colorblind-safe
- **vlag**: Similar to coolwarm, more saturated

**center=0**: Where to center the colormap

- **0**: Natural for correlations (-1 to +1)
- Makes positive/negative clear

**square=True**: Makes cells square

- Easier to read
- More professional appearance

**Reading a Correlation Heatmap:**

**Strong Positive (Dark Red):**

- Revenue & Marketing_Spend: 0.86
- **Action**: Continue investing in marketing

**Strong Negative (Dark Blue):**

- (If present) indicates inverse relationships
- Example: Price & Volume might be -0.65

**Weak Correlations (White/Light):**

- Employee_Satisfaction & Revenue: 0.15
- **Interpretation**: No direct link (may be indirect through other variables)

**Surprising Patterns:**

- Customer_Satisfaction & Revenue: Only 0.32
- **Question**: Why isn't satisfaction driving revenue more?
- **Possible answer**: Long-term effect not captured in current period data

**Advanced Heatmap Techniques**

**Masking the Upper Triangle (Avoid Redundancy):**

Since correlation matrices are symmetric, showing both triangles is redundant:


In [ ]:
import numpy as np

# Generate mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr_matrix,
            mask=mask,            # Hide upper triangle
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={'shrink': 0.8},
            ax=ax)

ax.set_title('Correlation Heatmap (Lower Triangle)', 
             fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()


**Output:**

*[Heatmap: "Correlation Heatmap (Lower Triangle)"]*

**Benefits:**

- Cleaner, more professional
- Eliminates distraction
- Saves space for larger variable sets

**Highlighting Strong Correlations:**


In [ ]:
# Create custom annotations to highlight strong correlations
annot_array = corr_matrix.values.copy()
annotations = []

for i in range(len(annot_array)):
    row_annot = []
    for j in range(len(annot_array[i])):
        val = annot_array[i, j]
        if abs(val) > 0.7 and i != j:  # Strong correlation, not diagonal
            row_annot.append(f'{val:.2f}*')  # Add asterisk
        else:
            row_annot.append(f'{val:.2f}')
    annotations.append(row_annot)

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr_matrix,
            annot=np.array(annotations),
            fmt='',  # Don't format, we already did
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=1,
            ax=ax)

ax.set_title('Correlation Heatmap (* indicates |r| > 0.7)', 
             fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()


**Output:**

*[Heatmap: "Correlation Heatmap (* indicates |r| > 0.7)"]*

**Pivot Table Heatmaps - Two-Way Categorical Analysis**

**Beyond Correlation - Categorical Relationships:**

Heatmaps aren't just for correlations. They excel at showing **values across two categorical dimensions**:

- Average sales by Region × Quarter
- Customer count by Segment × Industry
- Satisfaction scores by Department × Tenure

**Creating Pivot Table Heatmaps:**


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Generate sales data by region and quarter
np.random.seed(42)

regions = ['North', 'South', 'East', 'West']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

data_list = []
for region in regions:
    for quarter in quarters:
        # Different regions have different patterns
        if region == 'East':
            base = 85000
        elif region == 'West':
            base = 78000
        elif region == 'North':
            base = 82000
        else:
            base = 75000
        
        # Seasonal variation by quarter
        if quarter == 'Q4':
            seasonal = 15000  # Holiday boost
        elif quarter == 'Q1':
            seasonal = -5000  # Post-holiday slump
        else:
            seasonal = 0
        
        sales = base + seasonal + np.random.normal(0, 5000)
        data_list.append({
            'Region': region,
            'Quarter': quarter,
            'Sales': sales
        })

df_sales = pd.DataFrame(data_list)

# Create pivot table
pivot_table = df_sales.pivot_table(values='Sales', 
                                    index='Region', 
                                    columns='Quarter',
                                    aggfunc='mean')

# Create heatmap
fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(pivot_table,
            annot=True,
            fmt='.0f',  # No decimals for currency
            cmap='YlGnBu',  # Yellow to Green to Blue
            linewidths=2,
            linecolor='white',
            cbar_kws={'label': 'Average Sales ($)'},
            ax=ax)

ax.set_title('Average Sales by Region and Quarter - 2024', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Quarter', fontsize=12)
ax.set_ylabel('Region', fontsize=12)

plt.tight_layout()
plt.show()


**Output:**

*[Heatmap: "Average Sales by Region and Quarter - 2024", x: Quarter, y: Region]*

**Reading Two-Way Heatmaps:**

**Hotspots (Darker colors):**

- East + Q4: Highest sales ($100K)
- **Insight**: East region + Holiday season = best performance

**Cold spots (Lighter colors):**

- South + Q1: Lowest sales ($70K)
- **Insight**: South struggles post-holiday, needs Q1 promotion

**Patterns:**

- All regions: Q4 is best (holiday effect)
- East region: Consistently outperforms (across all quarters)

**Business Actions:**

- Allocate more inventory to East in Q4
- Create Q1 promotions for South
- Study East's practices to replicate elsewhere

**Clustermaps - Discovering Hidden Groupings**

**The Pattern Recognition Problem:**

Sometimes relationships aren't obvious from raw order. Clustermaps **automatically reorder** rows and columns to group similar variables together.

**What Clustermaps Reveal:**

- Which variables behave similarly
- Natural groupings in your metrics
- Redundant variables (measure same thing)
- Hierarchical relationships

**Creating a Clustermap:**


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Generate diverse business metrics
np.random.seed(42)
n = 150

df = pd.DataFrame({
    # Financial metrics (should cluster)
    'Revenue': np.random.normal(100000, 20000, n),
    'Profit': np.random.normal(25000, 8000, n),
    'Cash_Flow': np.random.normal(30000, 10000, n),
    
    # Customer metrics (should cluster)
    'Customer_Count': np.random.poisson(200, n),
    'Customer_Satisfaction': np.random.uniform(3.5, 5.0, n),
    'Net_Promoter_Score': np.random.normal(45, 15, n),
    
    # Operational metrics (should cluster)
    'Employee_Count': np.random.randint(20, 100, n),
    'Productivity_Score': np.random.normal(75, 12, n),
    'Efficiency_Rating': np.random.uniform(65, 95, n),
    
    # Marketing metrics (should cluster)
    'Marketing_Spend': np.random.normal(15000, 5000, n),
    'Lead_Generation': np.random.poisson(50, n),
    'Conversion_Rate': np.random.uniform(0.15, 0.35, n)
})

# Create realistic relationships within clusters
df['Profit'] = 0.25 * df['Revenue'] + np.random.normal(0, 3000, n)
df['Cash_Flow'] = 0.30 * df['Revenue'] + np.random.normal(0, 4000, n)
df['Net_Promoter_Score'] = 20 + 6 * df['Customer_Satisfaction'] + np.random.normal(0, 5, n)
df['Productivity_Score'] = 50 + 0.25 * df['Employee_Count'] + np.random.normal(0, 8, n)
df['Conversion_Rate'] = 0.15 + 0.000003 * df['Marketing_Spend'] + np.random.normal(0, 0.03, n)

# Calculate correlation
corr_matrix = df.corr()

# Create clustermap
g = sns.clustermap(corr_matrix,
                   annot=True,
                   fmt='.2f',
                   cmap='coolwarm',
                   center=0,
                   figsize=(14, 14),
                   linewidths=0.5,
                   cbar_kws={'label': 'Correlation Coefficient'})

g.fig.suptitle('Clustered Correlation Heatmap - Business Metrics', 
               fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()


**Output:**

*[Cluster heatmap: "Clustered Correlation Heatmap - Business Metrics"]*

**Reading a Clustermap:**

**Dendrograms (Tree structures on sides):**

- Show how variables are grouped
- Shorter branches = more similar
- Longer branches = less similar

**Reordered Matrix:**

- Variables automatically rearranged
- Similar variables placed adjacent
- Reveals natural groupings

**Business Insights from Clusters:**

**Cluster 1: Financial Metrics**

- Revenue, Profit, Cash_Flow group together
- **Insight**: These move together (as expected)
- **Action**: Can use Revenue as proxy for financial health

**Cluster 2: Customer Metrics**

- Customer_Satisfaction, NPS strongly linked
- **Insight**: Satisfied customers promote us
- **Action**: Focus on satisfaction, NPS will follow

**Cluster 3: Operational Metrics**

- Employee_Count, Productivity separate from others
- **Insight**: Operations independent of customer metrics
- **Question**: Should they be? Investigate connection

**Multicollinearity and Variable Selection**

**The Redundancy Problem:**

In predictive modeling or analysis, highly correlated independent variables cause problems:

- **Multicollinearity**: When predictors correlate with each other
- **Result**: Unstable models, unclear which variable matters
- **Solution**: Remove redundant variables

**Using Heatmaps to Identify Redundancy:**


In [ ]:
# Identify highly correlated variable pairs (excluding diagonal)
high_corr_pairs = []
threshold = 0.85

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) >= threshold:
            high_corr_pairs.append({
                'Variable 1': corr_matrix.columns[i],
                'Variable 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

df_high_corr = pd.DataFrame(high_corr_pairs)
print("\nHighly Correlated Variable Pairs (|r| >= 0.85):")
print(df_high_corr)


**Output:**
```

Highly Correlated Variable Pairs (|r| >= 0.85):
Empty DataFrame
Columns: []
Index: []
```

**Business Decision Framework:**

If Revenue and Cash_Flow correlate at r=0.92:
1. **Keep one, drop the other** for modeling
2. **Which to keep?**
   - The one easier to measure
   - The one more relevant to the business question
   - The one with less measurement error

**Example:**

- Revenue and Profit: r=0.88
- **For sales forecasting**: Keep Revenue (direct measure)
- **For profitability analysis**: Keep Profit (more relevant)

**Color Scheme Selection for Different Contexts**

**Choosing the Right Colors Matters:**

Different data types and questions require different color schemes:

**Diverging (Best for Correlations):**


In [ ]:
# Data centered at 0, both positive and negative values
cmap='coolwarm'  # Blue (negative) → White (zero) → Red (positive)
cmap='RdBu_r'    # Red → Blue reversed
cmap='PiYG'      # Pink → Yellow → Green


**Sequential (Best for One-directional Values):**


In [ ]:
# All positive values, low to high
cmap='YlGnBu'    # Yellow → Green → Blue
cmap='Blues'     # Light blue → Dark blue
cmap='Reds'      # Light red → Dark red


**Qualitative (Best for Categories):**


In [ ]:
# Distinct categories, no order
cmap='Set1'
cmap='Set2'
cmap='tab10'


**Business Context Examples:**

**Financial Performance (diverging):**

- Negative (loss) = Red
- Zero (break-even) = White
- Positive (profit) = Green


In [ ]:
cmap='RdYlGn'  # Red-Yellow-Green


**Sales Performance (sequential):**

- All positive values
- Low = Light, High = Dark


In [ ]:
cmap='YlOrRd'  # Yellow-Orange-Red (heat map feel)


**Customer Satisfaction (sequential):**

- All positive (1-5 scale)
- Low satisfaction = Red, High = Green


In [ ]:
cmap='RdYlGn'  # But not centered at 0, normalized to data range


### Lab Session
**Lab 4: Correlation and Heatmap Analysis**

**Objective:** Master correlation analysis and heatmap visualization to uncover complex relationships in multivariate business data and communicate insights through color-coded matrices.

**Scenario:** You're the Senior Data Analyst at "MarketPro Analytics," a marketing analytics firm. Your client, a major e-commerce company, has provided comprehensive data on their operations. The CMO is struggling to understand which metrics drive success and where to focus improvement efforts. Your task is to create correlation analyses and heatmaps that reveal the hidden relationships in their complex business data and provide clear, actionable insights for strategic planning.

**Pre-Lab Setup:**

1. Create file: `M3L04_YourName_Heatmaps.py`
2. Import and configure:


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_theme(style="white", context="talk")


3. Create output folder: "Lab4_Outputs"

**Part A: Comprehensive Business Metrics Correlation Analysis (30 points)**

**Context:** The client wants to understand how their various business metrics relate to each other to identify key drivers of success.

**Data:**


In [ ]:
np.random.seed(42)
n = 250  # 250 days of data

# Generate comprehensive e-commerce metrics
df_metrics = pd.DataFrame({
    'Daily_Revenue': np.random.normal(50000, 12000, n),
    'Website_Traffic': np.random.poisson(15000, n),
    'Conversion_Rate': np.random.uniform(0.02, 0.08, n),
    'Avg_Order_Value': np.random.normal(85, 20, n),
    'Marketing_Spend': np.random.normal(3000, 800, n),
    'Email_Open_Rate': np.random.uniform(0.15, 0.35, n),
    'Social_Media_Engagement': np.random.poisson(500, n),
    'Customer_Service_Tickets': np.random.poisson(45, n),
    'Product_Returns': np.random.poisson(25, n),
    'Page_Load_Time': np.random.uniform(1.5, 4.5, n),
    'Mobile_Traffic_Pct': np.random.uniform(0.45, 0.75, n),
    'Customer_Satisfaction': np.random.uniform(3.8, 4.8, n)
})

# Create realistic relationships
df_metrics['Daily_Revenue'] = (
    10000 +
    1.8 * df_metrics['Website_Traffic'] +
    200000 * df_metrics['Conversion_Rate'] +
    300 * df_metrics['Avg_Order_Value'] +
    2.5 * df_metrics['Marketing_Spend'] +
    -2000 * df_metrics['Page_Load_Time'] +
    np.random.normal(0, 5000, n)
)

df_metrics['Conversion_Rate'] = (
    0.03 +
    0.00000015 * df_metrics['Website_Traffic'] +
    0.01 * df_metrics['Email_Open_Rate'] +
    -0.003 * df_metrics['Page_Load_Time'] +
    0.002 * df_metrics['Customer_Satisfaction'] +
    np.random.normal(0, 0.005, n)
)
df_metrics['Conversion_Rate'] = df_metrics['Conversion_Rate'].clip(0.01, 0.10)

df_metrics['Customer_Satisfaction'] = (
    3.0 +
    0.3 * (5 - df_metrics['Page_Load_Time']) +
    -0.015 * df_metrics['Customer_Service_Tickets'] +
    -0.020 * df_metrics['Product_Returns'] +
    np.random.normal(0, 0.2, n)
)
df_metrics['Customer_Satisfaction'] = df_metrics['Customer_Satisfaction'].clip(1, 5)


**Tasks:**

1. **Create comprehensive correlation heatmap (12 points):**

   - Calculate full correlation matrix
   - Figure size: (16, 14)
   - Use 'coolwarm' colormap centered at 0
   - Annotate with correlation values (2 decimals)
   - Square cells with white lines (linewidth=1)
   - Add colorbar with label "Correlation Coefficient"
   - Title: "E-Commerce Business Metrics - Correlation Analysis"
   - Make text readable (appropriate font sizes)

2. **Create lower triangle version (8 points):**

   - Same heatmap but mask upper triangle
   - Eliminates redundancy
   - Cleaner professional appearance
   - Save as: `M3L04_YourName_FullCorrelation.png` (full) and `M3L04_YourName_LowerTriangle.png` (masked)

3. **Identify and document key insights (10 points):**

   - Find the THREE strongest positive correlations (excluding diagonal)
   - Find the THREE strongest negative correlations
   - Create a summary report in comments:
     * List these correlations with values
     * Explain what each relationship means for business
     * Identify surprising or counter-intuitive findings
     * Recommend which metrics to focus on for improving revenue
     * Identify any potential multicollinearity issues (r > 0.85)

**Part B: Revenue Driver Deep Dive (25 points)**

**Context:** The CMO specifically wants to know which factors most strongly influence revenue. Create a focused analysis on revenue correlations.

**Tasks:**

1. **Create revenue-focused correlation visualization (15 points):**

   - Extract correlations of all variables with 'Daily_Revenue'
   - Create a horizontal bar plot showing these correlations
   - Sort by correlation strength (highest to lowest by absolute value)
   - Color bars: Green for positive, Red for negative
   - Figure size: (12, 8)
   - Title: "Revenue Correlation Analysis - Key Drivers Identified"
   - Add vertical reference lines at -0.3, 0, +0.3, +0.6 (weak, none, moderate, strong thresholds)
   - Annotate bars with correlation values

2. **Statistical significance and interpretation (10 points):**

   - Calculate correlation with p-values using scipy.stats.pearsonr
   - In comments, document:
     * Top 3 revenue drivers (highest positive correlations)
     * Whether these correlations are statistically significant (p < 0.05)
     * Which metrics have surprisingly weak correlation with revenue
     * Actionable recommendations for increasing revenue based on findings
   - Create a text annotation on the plot highlighting the #1 revenue driver
   - Save as: `M3L04_YourName_RevenueDrivers.png`

**Part C: Marketing Performance Pivot Heatmap (25 points)**

**Context:** The marketing team runs campaigns across different channels (Email, Social, PPC, Display) in different customer segments (New, Returning, VIP). They need to see performance by Channel × Segment.

**Data:**


In [ ]:
np.random.seed(42)

channels = ['Email', 'Social Media', 'PPC', 'Display Ads']
segments = ['New Customers', 'Returning', 'VIP']

marketing_data = []
for channel in channels:
    for segment in segments:
        # Different channels perform differently with different segments
        if channel == 'Email' and segment == 'VIP':
            roi = np.random.normal(4.5, 0.5, 30)  # Email works great for VIP
        elif channel == 'Social Media' and segment == 'New Customers':
            roi = np.random.normal(3.8, 0.6, 30)  # Social good for acquisition
        elif channel == 'PPC' and segment == 'Returning':
            roi = np.random.normal(3.5, 0.4, 30)
        elif channel == 'Display Ads':
            roi = np.random.normal(2.2, 0.5, 30)  # Display generally weaker
        else:
            roi = np.random.normal(3.0, 0.5, 30)
        
        for r in roi:
            marketing_data.append({
                'Channel': channel,
                'Segment': segment,
                'ROI': max(r, 0.5)  # Ensure positive ROI
            })

df_marketing = pd.DataFrame(marketing_data)


**Tasks:**

1. **Create pivot table heatmap (15 points):**

   - Pivot: Channels as rows, Segments as columns, Average ROI as values
   - Figure size: (10, 6)
   - Use 'RdYlGn' colormap (Red=low, Yellow=medium, Green=high)
   - Annotate with ROI values (1 decimal place)
   - Add colorbar labeled "Average ROI ($return per $1 spent)"
   - Title: "Marketing ROI by Channel and Customer Segment"
   - Thick white lines between cells (linewidth=2)

2. **Add conditional formatting context (5 points):**

   - Calculate overall average ROI
   - Add subtitle text below title: "Overall Average ROI: $X.XX"
   - In annotations, mark cells with * if ROI > overall average + 0.5
   - This highlights exceptional performance

3. **Business recommendations (5 points):**

   - In code comments, identify:
     * Best performing Channel-Segment combination
     * Worst performing combination
     * Where to increase marketing budget
     * Where to decrease or eliminate spending
     * Unexpected patterns or opportunities
   - Save as: `M3L04_YourName_MarketingROI.png`

**Part D: Customer Behavior Clustermap (20 points)**

**Context:** The product team has metrics on customer behavior but doesn't know which behaviors cluster together or which customer segments behave similarly.

**Data:**


In [ ]:
np.random.seed(42)

# Customer behavior metrics
behaviors = pd.DataFrame({
    'Purchase_Frequency': np.random.gamma(2, 2, 200),
    'Avg_Session_Duration': np.random.normal(8, 3, 200),
    'Pages_Per_Session': np.random.poisson(5, 200),
    'Cart_Abandonment_Rate': np.random.uniform(0.3, 0.8, 200),
    'Wishlist_Usage': np.random.poisson(3, 200),
    'Review_Activity': np.random.poisson(2, 200),
    'Social_Shares': np.random.poisson(1, 200),
    'Coupon_Usage_Rate': np.random.uniform(0.1, 0.6, 200),
    'Customer_Service_Contact': np.random.poisson(1, 200),
    'Mobile_App_Usage': np.random.uniform(0.2, 0.9, 200)
})

# Create some natural relationships
behaviors['Purchase_Frequency'] = (
    behaviors['Purchase_Frequency'] * 
    (1 + 0.3 * behaviors['Wishlist_Usage'] / behaviors['Wishlist_Usage'].max())
)

behaviors['Review_Activity'] = (
    behaviors['Review_Activity'] + 
    0.3 * behaviors['Purchase_Frequency']
)

behaviors['Social_Shares'] = (
    behaviors['Social_Shares'] + 
    0.5 * behaviors['Review_Activity']
)

df_behaviors = behaviors


**Tasks:**

1. **Create clustermap (15 points):**

   - Calculate correlation matrix
   - Create clustermap with:
     * Figure size: (14, 14)
     * 'coolwarm' colormap centered at 0
     * Annotations (2 decimals)
     * Appropriate dendrogram ratio
   - Title: "Customer Behavior Clustering Analysis"
   - Both row and column clustering enabled

2. **Interpret clusters (5 points):**

   - In detailed comments, identify:
     * How many natural behavior clusters exist?
     * Which behaviors group together? (Name the clusters meaningfully)
     * Example: "Engagement Cluster: Review_Activity, Social_Shares, Wishlist_Usage"
     * Which behaviors are independent/isolated?
     * What do these clusters tell us about customer types?
     * How can marketing use this segmentation?
   - Save as: `M3L04_YourName_BehaviorClustermap.png`

**Bonus Challenge (+30 points):**

**Create a Comprehensive Executive Dashboard with Multiple Heatmaps:**

Build a figure with 4 subplots (2×2) showing:

1. **Top-left: Correlation heatmap of top 6 business metrics**
   - Select the 6 most important metrics
   - Lower triangle only
   - Highlight strong correlations

2. **Top-right: Time series heatmap**
   - Create data: Metrics (rows) × Weeks (columns)
   - Show how metrics evolved over 12 weeks
   - Use sequential colormap

3. **Bottom-left: Comparison heatmap**
   - This Year vs Last Year performance
   - Metrics as rows, Year as columns
   - Show % change with diverging colors

4. **Bottom-right: Priority matrix**
   - Impact (Low/Medium/High) × Effort (Low/Medium/High)
   - Place top 9 initiatives in appropriate cells
   - Color by priority level

**Requirements:**

- Figure size: (20, 18)
- Consistent, professional styling
- Overall title: "MarketPro Analytics - Executive Dashboard"
- Each subplot clearly labeled
- Annotations where appropriate
- Color schemes selected for context
- Save as: `M3L04_YourName_ExecutiveDashboard.png`

**Deliverables:**

1. Python file: `M3L04_YourName_Heatmaps.py`
2. PNG files (minimum 5):
   - `M3L04_YourName_FullCorrelation.png`
   - `M3L04_YourName_LowerTriangle.png`
   - `M3L04_YourName_RevenueDrivers.png`
   - `M3L04_YourName_MarketingROI.png`
   - `M3L04_YourName_BehaviorClustermap.png`
   - `M3L04_YourName_ExecutiveDashboard.png` (bonus)

**Grading Rubric:**

- Part A (Correlation Analysis): 30 points
- Part B (Revenue Drivers): 25 points
- Part C (Marketing Pivot): 25 points
- Part D (Clustermap): 20 points
- Bonus (Dashboard): +30 points

**Success Criteria:**

- [ ] Correlation values calculated correctly
- [ ] Appropriate color schemes selected for data context
- [ ] Annotations clear and readable
- [ ] Business insights documented in detail
- [ ] Multicollinearity issues identified
- [ ] Clustermaps reveal meaningful groupings
- [ ] Pivot heatmaps show actionable patterns
- [ ] All visualizations properly labeled with titles and colorbars
- [ ] High-resolution professional outputs

**Heatmap Best Practices Checklist:**

- [ ] Choose colormap appropriate for data type (diverging vs sequential)
- [ ] Center diverging colormaps at meaningful value (usually 0)
- [ ] Always annotate correlation heatmaps with values
- [ ] Use appropriate number formats (correlations: .2f, money: .0f)
- [ ] Include colorbar with descriptive label
- [ ] Make cells square for easier reading
- [ ] Use lines between cells for clarity
- [ ] Consider masking upper triangle for correlations
- [ ] Ensure text is readable (font size, contrast)
- [ ] Add meaningful titles that explain what's shown

**Common Mistakes to Avoid:**

- Using rainbow/jet colormap (not perceptually uniform)
- Forgetting to center diverging colormaps
- Too many variables (>15 becomes unreadable)
- Poor color contrast with annotations
- Missing colorbar or unlabeled colorbar
- Not interpreting business meaning of correlations
- Confusing correlation with causation
- Ignoring statistical significance

**Professional Tips:**

- Strong correlation doesn't mean causation
- Always consider lag effects (marketing → sales may take time)
- Look for unexpected weak correlations (why isn't X related to Y?)
- Use clustermaps to discover natural groupings
- Pivot heatmaps excellent for two-way categorical analysis
- Document threshold for "strong" correlation in your context
- Consider removing redundant highly-correlated variables for modeling

---

**End of Module 3: Statistical Visualization with Seaborn**

**Key Takeaways:**

- Seaborn provides high-level statistical visualizations with beautiful defaults
- Distribution plots (histogram, KDE, box, violin) reveal data characteristics
- Relationship plots (scatter with regression, joint, pair) uncover correlations
- Categorical plots (count, bar, box, violin, point) compare groups effectively
- Heatmaps and clustermaps visualize multivariate relationships compactly
- Always interpret statistical findings in business context
- Correlation does not imply causation
